# SanskritGPT-Vedic: Accented Model Training (Hyper-Precision)

This notebook trains a high-precision Transformer model on the **accented Vedic dataset**.
It follows the "Epic" training philosophy: hyper-precision monitoring, auto-best-model persistence, and optimized architectures to prevent overfitting on small datasets.

### Key Technical specs:
- **Architecture**: 11.5M Parameters (6 Layers, 8 Heads, 320 Embedding size)
- **Tokenizer**: Vedic Unigram (8,000 vocab)
- **Monitoring**: Live verse generation every 500-1000 steps.
- **Storage**: Space-optimized (Local /content training, final sync to Drive).

In [1]:
# 1. Hardware Check & Environment Setup
import os
import torch
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print("Running in Google Colab")
except:
    IN_COLAB = False
    print("Running Locally")

!nvidia-smi

Mounted at /content/drive
Running in Google Colab
Sun Apr  5 03:42:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A 

In [2]:
# 2. Install Dependencies
!pip install -q transformers datasets tokenizers sentencepiece accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00


In [3]:
# 3. Path Configuration
if IN_COLAB:
    # Update PROJECT_ROOT to match your actual Drive path structure
    PROJECT_ROOT = '/content/drive/MyDrive/Ved/SanskritGPT-Vedic'
    LOCAL_OUTPUT_DIR = '/content/model_checkpoints'
    DRIVE_BACKUP_DIR = f'{PROJECT_ROOT}/model_output_accented'
else:
    PROJECT_ROOT = '..'
    LOCAL_OUTPUT_DIR = '../model_output_accented'
    DRIVE_BACKUP_DIR = LOCAL_OUTPUT_DIR

DATA_FILE = f'{PROJECT_ROOT}/data/veda_train_accented.txt'
TOKENIZER_DIR = f'{LOCAL_OUTPUT_DIR}/tokenizer'
LOG_FILE = f'{LOCAL_OUTPUT_DIR}/training_log.txt'

os.makedirs(LOCAL_OUTPUT_DIR, exist_ok=True)
os.makedirs(TOKENIZER_DIR, exist_ok=True)

if not os.path.exists(DATA_FILE):
    print(f"!!! WARNING: DATA_FILE NOT FOUND AT {DATA_FILE} !!!")
    print("Please check your PROJECT_ROOT path in the cell above.")
else:
    print(f"Project Root: {PROJECT_ROOT}")
    print(f"Data File Found: {DATA_FILE}")

Project Root: /content/drive/MyDrive/Ved/SanskritGPT-Vedic
Data File Found: /content/drive/MyDrive/Ved/SanskritGPT-Vedic/data/veda_train_accented.txt


In [4]:
# 4. Data Analysis
from collections import Counter
import re

if os.path.exists(DATA_FILE):
    with open(DATA_FILE, 'r', encoding='utf-8') as f:
        lines = [l.strip() for l in f if l.strip()]

    print(f"Total non-empty lines: {len(lines)}")

    # Detect tags like <RIG>, <YAJUR>, etc.
    tags = []
    for line in lines:
        match = re.search(r'<[A-Z]+>', line)
        if match: tags.append(match.group())

    tag_counts = Counter(tags)
    print("\nTag Distribution (First 1000 lines):")
    for tag, count in tag_counts.items():
        print(f"{tag}: {count} lines")

    avg_len = sum(len(l) for l in lines) / len(lines)
    print(f"\nAverage line length: {avg_len:.1f} characters")
    print("Sample line:", lines[0] if lines else "No lines found")
else:
    print("Cannot analyze: DATA_FILE not found.")

Total non-empty lines: 20015

Tag Distribution (First 1000 lines):
<RIG>: 10403 lines
<YAJUR>: 3719 lines
<ATHARVA>: 5893 lines

Average line length: 126.6 characters
Sample line: <RIG> अ॒ग्निमी॑ळे पु॒रोहि॑तं य॒ज्ञस्य॑ दे॒वमृ॒त्विज॑म् । होता॑रं रत्न॒धात॑मम् ॥ <eos>


In [5]:
# 5. Unigram Tokenizer Training (from scratch)
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers, decoders
from transformers import PreTrainedTokenizerFast

VOCAB_SIZE = 8000
SPECIAL_TOKENS = ["<s>", "<pad>", "</s>", "<unk>", "<mask>", "<RIG>", "<YAJUR>", "<ATHARVA>", "<SAM>", "<eos>"]

if 'lines' in locals():
    print(f"Training Unigram Tokenizer (Vocab={VOCAB_SIZE})...")
    tokenizer_obj = Tokenizer(models.Unigram())
    tokenizer_obj.normalizer = normalizers.Sequence([normalizers.NFKC()])
    tokenizer_obj.pre_tokenizer = pre_tokenizers.Metaspace(replacement="▁", prepend_scheme="always")
    tokenizer_obj.decoder = decoders.Metaspace(replacement="▁", prepend_scheme="always")

    trainer = trainers.UnigramTrainer(
        vocab_size=VOCAB_SIZE,
        special_tokens=SPECIAL_TOKENS,
        unk_token="<unk>"
    )

    tokenizer_obj.train_from_iterator(lines, trainer=trainer)
    tokenizer_obj.save(f"{TOKENIZER_DIR}/tokenizer.json")

    tokenizer = PreTrainedTokenizerFast(
        tokenizer_object=tokenizer_obj,
        bos_token="<s>",
        eos_token="<eos>",
        unk_token="<unk>",
        pad_token="<pad>",
        mask_token="<mask>"
)
    tokenizer.save_pretrained(TOKENIZER_DIR)
    print("Tokenizer saved successfully.")
else:
    print("Missing 'lines' - please run data analysis cell first.")

Training Unigram Tokenizer (Vocab=8000)...
Tokenizer saved successfully.


In [6]:
# 6. Model Architecture & Logging Callback
from transformers import GPT2Config, GPT2LMHeadModel, Trainer, TrainingArguments, TrainerCallback, EarlyStoppingCallback, DataCollatorForLanguageModeling
from datasets import load_dataset, concatenate_datasets

if 'tokenizer' in locals():
    # 11.5M Parameter Configuration
    config = GPT2Config(
        vocab_size=len(tokenizer),
        n_positions=512,
        n_ctx=512,
        n_embd=320,
        n_layer=6,
        n_head=8,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        resid_pdrop=0.1,
        embd_pdrop=0.1,
        attn_pdrop=0.1
)
    model = GPT2LMHeadModel(config)
    print(f"Model Parameters: {model.num_parameters()/1e6:.1f}M")

    class DetailedLogCallback(TrainerCallback):
        def __init__(self, tokenizer, log_file):
            self.tokenizer = tokenizer
            self.log_file = log_file

        def on_log(self, args, state, control, logs=None, **kwargs):
            if state.is_world_process_zero and logs and "loss" in logs:
                model = kwargs['model']
                device = next(model.parameters()).device
                model.eval()
                prompts = ["<RIG> ", "<YAJUR> ", "<ATHARVA> "]
                samples = []
                for p in prompts:
                    inputs = self.tokenizer(p, return_tensors="pt").to(device)
                    with torch.no_grad():
                        outputs = model.generate(
                            inputs.input_ids, max_length=50, do_sample=True, temperature=0.8, top_p=0.95,
                            eos_token_id=self.tokenizer.eos_token_id, pad_token_id=self.tokenizer.pad_token_id
                        )
                    samples.append(self.tokenizer.decode(outputs[0], skip_special_tokens=False))
                model.train()

                log_entry = f"Step {state.global_step} | Loss: {logs['loss']:.4f}\n"
                for i, p in enumerate(prompts):
                    log_entry += f"Sample {p.strip()}: {samples[i]}\n"
                log_entry += "="*50 + "\n"
                print(log_entry)
                with open(self.log_file, "a", encoding="utf-8") as f: f.write(log_entry)
else:
    print("Tokenizer not initialized.")

Model Parameters: 10.1M


In [8]:
# 7. Dataset Mapping & Training
if 'model' in locals() and os.path.exists(DATA_FILE):
    dataset = load_dataset("text", data_files={"train": DATA_FILE})
    def tokenize_fn(examples): return tokenizer(examples["text"], truncation=True, max_length=512)
    tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
    split = tokenized["train"].train_test_split(test_size=0.1, seed=42)

    # 2x Augmentation
    aug_train = concatenate_datasets([split["train"], split["train"]]).shuffle(seed=42)
    print(f"Training set size (augmented): {len(aug_train)}")

    training_args = TrainingArguments(
        output_dir=LOCAL_OUTPUT_DIR,
        num_train_epochs=30,
        per_device_train_batch_size=32,
        gradient_accumulation_steps=1,
        learning_rate=3e-5,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        logging_steps=500,
        eval_strategy="steps",
        eval_steps=1000,
        save_strategy="steps",
        save_steps=1000,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        fp16=torch.cuda.is_available(),
        report_to="none"
)

    trainer = Trainer(
        model=model,
        args=training_args,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
        train_dataset=aug_train,
        eval_dataset=split["test"],
        callbacks=[DetailedLogCallback(tokenizer, LOG_FILE), EarlyStoppingCallback(early_stopping_patience=5)]
)

    trainer.train()

    # Final Export & Drive Sync
    FINAL_MODEL_DIR = f"{LOCAL_OUTPUT_DIR}/final_model"
    trainer.save_model(FINAL_MODEL_DIR)
    tokenizer.save_pretrained(FINAL_MODEL_DIR)

    if IN_COLAB:
        os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
        print(f"Syncing final model and log to Drive: {DRIVE_BACKUP_DIR}")
        !cp -r {FINAL_MODEL_DIR}/* {DRIVE_BACKUP_DIR}/
        !cp {LOG_FILE} {DRIVE_BACKUP_DIR}/

Map:   0%|          | 0/20015 [00:00<?, ? examples/s]

Training set size (augmented): 36026


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
1000,7.249483,6.814328
2000,6.128487,6.036741
3000,5.843227,5.763486
4000,5.603134,5.545373
5000,5.420320,5.393428
6000,5.277660,5.279492
7000,5.164640,5.181143
8000,5.063327,5.100044
9000,4.966403,5.024410
10000,4.884863,4.962576


Step 500 | Loss: 8.1895
Sample <RIG>: <RIG>    ॒ व॑ भिः म॒घोन <eos>
Sample <YAJUR>: <YAJUR>  <eos>
Sample <ATHARVA>: <ATHARVA>   <eos>

Step 1000 | Loss: 7.2495
Sample <RIG>: <RIG> ा म्न् ॥न॒व॒॒॑ ।॒ अ <eos>
Sample <YAJUR>: <YAJUR> े॒॑न्तुं ॑ ।े <eos>
Sample <ATHARVA>: <ATHARVA>  ॒॒ं॒॑ । ॥ <eos>



...

Step 33500 | Loss: 4.0871
Sample <RIG>: <RIG> यस्य॑ व॒र्ति म॑क्ष॒ आ यद् मृ॑षां॒ मृ॑जद॒व्यास॑ । ये ज॒ति॒ यस्य॑ ते ने॒दं न ते वां॒ तु॑ता॒नव॑म् 
Sample <YAJUR>: <YAJUR> राज्ञी॑ मे॒ मूरै मे त्वा नि॒म्रदᳪ प्रोभे॑म । वां॒ मौय॑तां॒ नि मूलू॑नान् रर॒वाट् ॥ <eos>
Sample <ATHARVA>: <ATHARVA> विष्णुः॒ कद्म॑नु॒ परि॒ दिद्रु॒ण॑द् तृ॒ञ्जये॑ । अ॒धं कृ॒णोति॑ ॥ <eos>



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Syncing final model and log to Drive: /content/drive/MyDrive/Ved/SanskritGPT-Vedic/model_output_accented
